# K-Means

K-means clustering is a popular unsupervised machine learning algorithm used for grouping similar data points into k - clusters. Goal: to partition a given dataset into k (predefined) clusters.

Initialize:
- K: number of clusters
- Data: the input dataset
- Randomly select K initial centroids

Repeat:
- Assign each data point to the nearest centroid (based on Euclidean distance)
- Calculate the mean of each cluster to update its centroid
- Check if the centroids have converged (i.e., they no longer change)

Until:
- The centroids have converged
- The maximum number of iterations has been reached

Output:
- The final K clusters and their corresponding centroids

## Implementation using Numpy

In [3]:
import numpy as np 

class kMeans:
    def __init__(self, k, max_iters=100, tol=1e-4, seed=0):
        self.k = k
        self.max_iters = max_iters
        self.tol = tol
        self.seed = seed
        np.random.seed(seed)
        self.centroids = None
        self.labels = None

    def initialize(self, X):
        # randomly sample k data points as initial centroids
        centroids = X[np.random.choice(X.shape[0], size=self.k, replace=False)]
        return centroids
    
    def update_centroids(self, X, labels, old_centroids):
        new_centroids = np.zeros_like(self.centroids)
        for l in range(self.k):
            cluster_index = labels == l
            if np.sum(cluster_index) > 0:  # avoid empty cluster
                new_centroids[l] = X[cluster_index].mean(axis=0)
            else:
                new_centroids[l] = old_centroids[l]  # keep the same if no points assigned
        return new_centroids
    
    def update_centroids_vectorized(self, X, labels, old_centroids):
        N, D = X.shape

        # one-hot encoding for labels: (N, k)
        one_hot = np.zeros((N, self.k))
        one_hot[np.arange(N), labels] = 1

        # sum of samples per cluster: (k, D)
        sums = one_hot.T @ X # (k, N) @ (N, D) -> (k, D)
        # counts of samples per cluster: (k, )
        counts = one_hot.sum(axis=0) # (k, )
        # avoid division by zero
        # compute centroids: (k, D)
        centroids = sums / np.maximum(counts[:, None], 1)
        
        # fix empty centeroids by keeping the old ones
        empty = counts == 0
        centroids[empty, :] = old_centroids[empty, :]
        
        return centroids
    
    def fit(self, X):
        # initialize centroids
        centroids = self.initialize(X)

        # main loop
        for i in range(self.max_iters):
            # calculate distances between data points and centroids
            dists = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)  # (N, k)

            # assgin labels to the closest centroid
            labels = dists.argmin(axis=1)

            # update centroids for each cluster
            new_centroids = self.update_centroids_vectorized(X, labels, centroids)

            # check for convergence
            if np.linalg.norm(new_centroids - centroids) < self.tol:
                break

            centroids = new_centroids

        # update the final centroids and labels
        self.centroids = centroids
        self.labels = labels

In [4]:
N, D, k = 500, 10, 5
X = np.random.rand(N, D)

kmeans = kMeans(k=k)
kmeans.fit(X)
centroids, labels = kmeans.centroids, kmeans.labels
print("Centroids:\n", centroids)

Centroids:
 [[0.69723984 0.37614091 0.56122297 0.56091617 0.41980299 0.49983941
  0.77554871 0.3496149  0.65112317 0.50609147]
 [0.46357332 0.7513081  0.45474615 0.5539109  0.68767024 0.73501657
  0.48335319 0.48101136 0.50843491 0.45849354]
 [0.67289866 0.37953564 0.43565938 0.39808863 0.60059929 0.42369078
  0.2837654  0.60475666 0.34510938 0.37079379]
 [0.26649389 0.3647926  0.51153408 0.57913913 0.30466679 0.40559195
  0.2567156  0.4519407  0.60455329 0.61104618]
 [0.32009913 0.56616348 0.56983121 0.45392778 0.43741011 0.38213913
  0.76727321 0.67803326 0.32986608 0.54800947]]


## Vectorized centroid update

see `update_centroids_vectorized()` method.

## KMeans++

KMeans++ improve the above KMeans by better initializing/updating the centroids so that the algorithm can converge faster and avoid local minima. 

For initialization, instead of randomly and indenpendently picking k samples for centroids, KMeans++ sequentially selects centroids with probability propotional to the squared distance from existing centroids.

KMeans++ Initization:
- sample the first centroid from the data uniformly at random
- for each data sample, compute distance to nearest centroids: D(x)
- sample the next centroid with probability:
    $$p(x) = \frac{D(x)^2}{\sum D(x)^2}$$

    **Samples far away from existing centroids are more likely to be chosen**

- repeat until we have k centroids

The remaining for loop is same as KMeans.

In [5]:
class kMeansPlusPlus(kMeans):
    def initialize(self, X):
        """
        Args:
            X: (N, D)
        Returns:
            centroids: (k, D)
        """
        N, D = X.shape

        # randomly select the first centroid from the data points
        centroids = np.zeros((k, D))
        first_index = np.random.choice(N)
        centroids[0] = X[first_index]

        for i in range(1, k):
            # compute distances from existing centroids to all data points
            dists = np.linalg.norm(X[:, None, :] - centroids[None, :i, :], axis=2)  # (N, i)

            # find the minimum distance to any existing centroid for each data point
            min_dists = np.min(dists, axis=1)  # (N,)

            # select the next centroid with probability proportional to the square of the distance
            probs = min_dists ** 2
            probs /= probs.sum()

            next_index = np.random.choice(N, p=probs)
            centroids[i] = X[next_index]

        return centroids

In [6]:
kmeans = kMeansPlusPlus(k=k)
kmeans.fit(X)
centroids, labels = kmeans.centroids, kmeans.labels
print("Centroids:\n", centroids)

Centroids:
 [[0.42593575 0.63090233 0.74115475 0.71651011 0.48166871 0.39297163
  0.7171104  0.50717683 0.45572981 0.53913211]
 [0.66898343 0.25763405 0.49245542 0.50426477 0.36612575 0.45958524
  0.56931147 0.30470628 0.72949638 0.42843385]
 [0.61415767 0.54514601 0.65948291 0.44034355 0.64709308 0.53469927
  0.2598994  0.53534615 0.3476968  0.33966081]
 [0.33538257 0.3641815  0.39551277 0.38561212 0.38034072 0.39645096
  0.51773772 0.78262568 0.39516976 0.50967866]
 [0.43685883 0.59607283 0.19666004 0.4692334  0.57388353 0.66441994
  0.51924871 0.39440645 0.53393617 0.67535309]]
